In [1]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# torch.multiprocessing.set_start_method('spawn', force=True)
import time
import importlib
import torch.optim as optim
from torchsummary import summary
import os
import statistics

os.chdir("../")
from src.data_preparation import SimulatedDataset, collate_function, load_real_data
from src.transformers import MiniTransformer
import src.transformers as transformerFunctions
from src.transformers import init_weights_recursive
from src.transformers import print_parameters
from src.transformers import create_custom_mask, create_distance_to_end_matrix, create_pairwise_distance_matrix
from src.evaluation import calculate_bench1_loss, calculate_bench2_loss, calculate_repeat_loss, calculate_regression_loss, evaluate_mini_transformer
from src.statistical_testing import statistical_testing, print_p_values, plot_context_predindex_pair_effect, get_context_predindex_pair_effect
import torch.autograd.profiler as profiler
from sklearn.model_selection import KFold
device = torch.device("cpu")

In [5]:

if __name__ == '__main__':
    
    bench_repeat_loss_list = []
    regression_loss_total_list = []
    model_loss_total_list = []
    regression_loss_predindex_list = []
    model_loss_predindex_list = []
    bench1_loss_list = []
    bench2_loss_list = []
    bench2loss_predindex_list = []
    models = []
    


    # Hyperparameters
    # data_str = "ghq_b_sum"
    data_str = "ghq_sum"
    # data_str = "simulation"
    batch_size = 1          # Batch size for loading data
    dk = 1                  # d_k
    dv = 1                  # d_v
    nheads = 8             # number of heads
    ncum = 8                 # number of cumulants
    maxlen = 10             # maximum length of the sequence
    learning_rate = 1e-3
    lambda_l2 = 1e-3
    EPOCHS = 100
    target_sample_size = 7
    nrepp = 10
    seeds = [0, 1, 11, 42, 123, 999, 1337, 2025, 9999, 12345]
    
    k_for_cross_val = 10
    
    # load real data
    data, maxlen = load_real_data(data_str)
    
    
    # Create the KFold splitter:
    kf = KFold(n_splits=k_for_cross_val, shuffle=True, random_state=42)

    # Build a list of (train, eval) folds:
    folds = []
    for train_index, eval_index in kf.split(data):
        train_data = [data[i] for i in train_index]
        eval_data  = [data[i] for i in eval_index]
        folds.append((train_data, eval_data))
    
    
    
    for (train, val) in folds:
        
        # Set the random seed for reproducibility
        torch.manual_seed(42)


        
        n = len(train)
        p = train[0].shape[1]
        print("Train size: ", len(train), "\n")
        print("Val size: ", len(val), "\n")
        
        train_dataset = train
        eval_dataset = val
        predindex = 9

        

        mask = create_custom_mask(maxlen, device)
        distance_to_end_matrix = create_distance_to_end_matrix(maxlen, device)
        pairwise_distance_matrix = create_pairwise_distance_matrix(maxlen, device)

    
        dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_function,  num_workers=0)
        
        model = MiniTransformer(p, nheads, dk, dv, ncum, mask, pairwise_distance_matrix, distance_to_end_matrix,  device)

        model.apply(init_weights_recursive)
        model.to(device)

        # model = torch.compile(model)
        # Start the timer
        start_time = time.time()

        # Define optimizer
        # optimizer = optim.Adam(model.parameters(), lr= learning_rate, weight_decay=lambda_l2)
        optimizer = optim.Adam(model.parameters(), lr= learning_rate)
        
        print("Number of Parameters", transformerFunctions.count_parameters(model))
        
        run_path = transformerFunctions.train_mini_transformer(model, dataloader, optimizer, lambda_l2, EPOCHS, device)


        # # Enable profiling
        # with profiler.profile() as prof:

        #     model.eval() 
        #     # Forward pass
        #     output = model(train_dataset.data)
        #     # Backward pass (this will profile the backward pass as well)
        #     output.backward(torch.ones_like(output))

        # # Print profiling results
        # print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))



        # End the timer
        end_time = time.time()

        # Calculate and print the execution time
        execution_time = end_time - start_time 
        print(f"Execution time: {execution_time:.6f} seconds")


        
        
        
        # Evaluate the model
        
        eval_dataloader = DataLoader(eval_dataset, batch_size=1, shuffle=True, collate_fn=collate_function, num_workers=0)
       
    
    
        dimave, bench1_loss = calculate_bench1_loss(train_dataset, eval_dataset)
        regression_loss_predindex, regression_loss_total = calculate_regression_loss(train_dataset, eval_dataset, predindex)
        model_loss_predindex, model_loss_total = evaluate_mini_transformer(eval_dataloader, model, predindex)
        
        
        print("baseline average loss: ", bench1_loss)

        
        # Evaluate the model
        if data_str == "simulation":
            bench2loss, bench2loss_predindex = calculate_bench2_loss(train_dataset, eval_dataset, dimave)
            print("baseline informed loss: ", bench2loss)
            bench2_loss_list.append(bench2loss)
            bench2loss_predindex_list.append(bench2loss_predindex)
            
        else:
            bench_repeat = calculate_repeat_loss(eval_dataset)    
            print("baseline repeat: ", bench_repeat)
            bench_repeat_loss_list.append(bench_repeat)
            
            
        print("regression loss total: ", regression_loss_total)
        print("model loss total: ", model_loss_total.item(), "\n") 
        
        
        if data_str == "simulation":
            print("baseline informed loss predindex: ", bench2loss_predindex)
            
        print("regression loss predindex: ", regression_loss_predindex)
        print("model loss predindex: ", model_loss_predindex.item()) 

        
        regression_loss_total_list.append(regression_loss_total)
        model_loss_total_list.append(model_loss_total.item())
        regression_loss_predindex_list.append(regression_loss_predindex)
        model_loss_predindex_list.append(model_loss_predindex.item())
        bench1_loss_list.append(bench1_loss)
        
        
        
# save the mean and std of these lists in text file in the format of mean ± std 
# os.chdir("./notebooks/results")



with open(f"./notebooks/{data_str}_results_n={n}.txt", "a") as f:
    # First line
    f.write(f"{data_str} results n = {n}\n")

    # Single-line writes with three-decimal formatting:
    f.write(f"baseline repeat: {statistics.mean(bench_repeat_loss_list):.3f} ± {statistics.stdev(bench_repeat_loss_list):.3f}\n")
    f.write(f"baseline average loss: {statistics.mean(bench1_loss_list):.3f} ± {statistics.stdev(bench1_loss_list):.3f}\n")
    if data_str == "simulation":
        f.write(f"baseline informed loss: {statistics.mean(bench2_loss_list):.3f} ± {statistics.stdev(bench2_loss_list):.3f}\n")
    f.write(f"regression loss total: {statistics.mean(regression_loss_total_list):.3f} ± {statistics.stdev(regression_loss_total_list):.3f}\n")
    f.write(f"model loss total: {statistics.mean(model_loss_total_list):.3f} ± {statistics.stdev(model_loss_total_list):.3f}\n\n")

    
    
    f.write(f"regression loss predindex: {statistics.mean(regression_loss_predindex_list):.3f} ± {statistics.stdev(regression_loss_predindex_list):.3f}\n")
    if data_str == "simulation":
        f.write(f"baseline informed loss predindex: {statistics.mean(bench2loss_predindex_list):.3f} ± {statistics.stdev(bench2loss_predindex_list):.3f}\n")
    f.write(f"model loss predindex: {statistics.mean(model_loss_predindex_list):.3f} ± {statistics.stdev(model_loss_predindex_list):.3f}\n")

    # Extra newlines at the end
    f.write("\n\n")


Train size:  790 

Val size:  88 

Number of Parameters 420
EPOCH 1:
avg_loss: 18.45028
EPOCH 2:
avg_loss: 14.49694
EPOCH 3:
avg_loss: 14.02481
EPOCH 4:
avg_loss: 13.77141
EPOCH 5:
avg_loss: 13.65091
EPOCH 6:
avg_loss: 13.56446
EPOCH 7:
avg_loss: 13.51648
EPOCH 8:
avg_loss: 13.48310
EPOCH 9:
avg_loss: 13.44897
EPOCH 10:
avg_loss: 13.43095
EPOCH 11:
avg_loss: 13.42099
EPOCH 12:
avg_loss: 13.39943
EPOCH 13:
avg_loss: 13.38104
EPOCH 14:
avg_loss: 13.33958
EPOCH 15:
avg_loss: 13.23828
EPOCH 16:
avg_loss: 13.16550
EPOCH 17:
avg_loss: 13.12614
EPOCH 18:
avg_loss: 13.08078
EPOCH 19:
avg_loss: 13.04116
EPOCH 20:
avg_loss: 13.00365
EPOCH 21:
avg_loss: 12.97818
EPOCH 22:
avg_loss: 12.95879
EPOCH 23:
avg_loss: 12.93442
EPOCH 24:
avg_loss: 12.90966
EPOCH 25:
avg_loss: 12.88592
EPOCH 26:
avg_loss: 12.85818
EPOCH 27:
avg_loss: 12.83151
EPOCH 28:
avg_loss: 12.78227
EPOCH 29:
avg_loss: 12.74006
EPOCH 30:
avg_loss: 12.70900
EPOCH 31:
avg_loss: 12.67528
EPOCH 32:
avg_loss: 12.66402
EPOCH 33:
avg_loss: 1